# CAP CDSS Agent - MedGemma 1.5 4B + LangGraph

**Community-Acquired Pneumonia Clinical Decision Support System**

A **8-node agentic pipeline** for CAP assessment following evidence-based CAP management principles.

> This notebook is generated by `_generate_full_pipeline_demo_notebook.py`. Do not edit manually.

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. *(Optional)* Add **GITHUB_TOKEN** for private repo install
4. **Run All** (Runtime -> Run all)

### Architecture
- **Model:** `google/medgemma-1.5-4b-it` (bfloat16, A100)
- **Framework:** LangGraph StateGraph (8 nodes + conditional routing)
- **Package:** `cap-agent` (pip-installable from GitHub)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/CAP-CDSS-MedGemma.git@main"
    except Exception:
        # Fall back to public install (will fail if repo is private)
        repo_url = "git+https://github.com/HP-00/CAP-CDSS-MedGemma.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet nest-asyncio
else:
    # Local development: assume pip install -e ".[dev]" already done
    print("Local environment detected. Ensure: pip install -e '.[dev]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## 0.5 Download CXR Images


In [ ]:
import os
import urllib.request

# Google's longitudinal CXR pair (same patient, clinical progression)
CXR_BASE = "https://storage.googleapis.com/hai-cd3-foundations-public-vault-entry/med_gemma/colab_example/cxr"
CXR_FILES = {
    "longitudinal_cxr_after.png": f"{CXR_BASE}/longitudinal_cxr_after.png",
    "longitudinal_cxr_before.png": f"{CXR_BASE}/longitudinal_cxr_before.png",
}

for filename, url in CXR_FILES.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
    else:
        print(f"Already exists: {filename}")

CXR_CURRENT = "longitudinal_cxr_after.png"
CXR_PRIOR = "longitudinal_cxr_before.png"

from PIL import Image
img = Image.open(CXR_CURRENT)
print(f"Current CXR: {img.size[0]}x{img.size[1]} {img.mode}")
img = Image.open(CXR_PRIOR)
print(f"Prior CXR: {img.size[0]}x{img.size[1]} {img.mode}")
print("CXR images ready")


## 1. Authentication & Imports


In [ ]:
import os
import json
import time

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    # Local: expects HF_TOKEN already in environment
    pass

# Verify token
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."

# Import the CAP agent package
from cap_agent.agent.graph import build_cap_agent_graph
from cap_agent.agent.state import build_initial_state
from cap_agent.data.synthetic import get_synthetic_case, get_synthetic_fhir_case
from cap_agent.models.medgemma import get_model_and_processor

print("Imports complete")


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    B = f"{ESC}[1m"       # Bold
    R = f"{ESC}[91m"      # Red
    Y = f"{ESC}[93m"      # Yellow
    G = f"{ESC}[92m"      # Green
    C = f"{ESC}[96m"      # Cyan
    X = f"{ESC}[0m"       # Reset

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{B}{C}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{X}")
    print(f"{Y}AI-generated observations for clinician review — not a substitute for clinical judgement.{X}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{B}{C}1. PATIENT{X}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {R}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){X}")
            else:
                print(f"  {R}Allergy: {a}{X}")
    else:
        print(f"  {G}No known drug allergies{X}")
    combos = demo.get("comorbidities", [])
    if combos:
        print(f"  Comorbidities: {', '.join(combos)}")
    if demo.get("pregnancy"):
        print(f"  {R}PREGNANT{X}")
    print(f"  Oral tolerance: {'Yes' if demo.get('oral_tolerance', True) else R + 'No' + X}")
    travel = demo.get("travel_history", [])
    if travel:
        print(f"  {Y}Travel: {', '.join(travel)}{X}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    sev_color = R if sev == "high" else (Y if sev == "moderate" else G)
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{B}{C}2. SEVERITY{X}")
    print(f"  {score_str}  {sev_color}{B}{sev.upper()}{X}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {Y}Missing: {', '.join(missing)}{X}")
    print()

    # 3. CXR
    s3 = sections.get("3_cxr", {})
    cxr = s3.get("findings", {})
    print(f"{B}{C}3. CHEST X-RAY{X}")
    for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
        f = cxr.get(finding, {})
        if not isinstance(f, dict):
            continue
        present = f.get("present", False)
        conf = f.get("confidence", "?")
        if present:
            loc = f.get("location", "")
            bbox = f.get("bounding_box")
            line = f"  {R}PRESENT{X} {finding} ({conf} confidence)"
            if loc:
                line += f" at {loc}"
            if bbox:
                line += f" [bbox: {bbox}]"
            print(line)
        else:
            print(f"  {G}absent{X}  {finding} ({conf} confidence)")
    iq = cxr.get("image_quality", {})
    if iq:
        print(f"  Quality: {iq.get('projection','?')} / {iq.get('rotation','?')} / {iq.get('penetration','?')}")
    longit = cxr.get("longitudinal_comparison")
    if longit:
        print(f"  {B}Longitudinal:{X}")
        for fn, ch in longit.items():
            if isinstance(ch, dict):
                print(f"    {fn}: {ch.get('change','?')} — {ch.get('description','')}")
    print()

    # 4. KEY BLOODS
    s4 = sections.get("4_key_bloods", {})
    labs = s4.get("values", {})
    print(f"{B}{C}4. KEY BLOODS{X}")
    for test, data in (labs or {}).items():
        if not isinstance(data, dict):
            continue
        val = data.get("value", "?")
        unit = data.get("unit", "")
        ref = data.get("reference_range", "")
        abn = data.get("abnormal_flag", False)
        color = R if abn else G
        flag = " ABNORMAL" if abn else ""
        print(f"  {color}{test}: {val} {unit}{flag}{X}  (ref: {ref})")
    print()

    # 5. CONTRADICTIONS
    s5 = sections.get("5_contradiction_alert", {})
    alerts = s5.get("alerts", [])
    informational = s5.get("informational", [])
    resolutions = s5.get("resolutions", [])
    n_total = s5.get("detected", 0)
    print(f"{B}{C}5. CONTRADICTION ALERTS{X}")
    if n_total == 0:
        print(f"  {G}No contradictions detected — findings concordant{X}")
    else:
        print(f"  {n_total} contradiction(s) detected")
        for c in alerts:
            conf = c.get("confidence", "?")
            badge_color = R if conf == "high" else Y
            print(f"  {badge_color}[{conf.upper()}]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
            if c.get("evidence_for"):
                print(f"    For: {c['evidence_for']}")
            if c.get("evidence_against"):
                print(f"    Against: {c['evidence_against']}")
            rec = c.get("recommendation")
            if rec:
                print(f"    {B}Recommendation:{X} {rec}")
        for c in informational:
            print(f"  {G}[LOW]{X} {c.get('rule_id','?')}: {c.get('pattern','')}")
        if resolutions:
            print(f"  {B}Resolutions:{X}")
            for r in resolutions:
                print(f"    {r[:200]}")
    print()

    # 6. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{B}{C}6. TREATMENT{X}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {Y}Allergy adj: {abx['allergy_adjustment']}{X}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    if abx.get("renal_adjustment"):
        print(f"  {Y}Renal: {abx['renal_adjustment']}{X}")
    cortico = s6.get("corticosteroid")
    if cortico:
        print(f"  Corticosteroid: {cortico}")
    steward = abx.get("stewardship_notes", [])
    for note in steward:
        print(f"  {Y}Stewardship: {note}{X}")
    inv = s6.get("investigations", {})
    if inv:
        print(f"  {B}Investigations:{X}")
        for name, detail in inv.items():
            if isinstance(detail, dict):
                rec = "Recommended" if detail.get("recommended") else "Not indicated"
                print(f"    {name}: {rec} — {detail.get('reasoning', '')[:100]}")
    print()

    # 7. DATA GAPS
    s7 = sections.get("7_data_gaps", {})
    gaps = s7.get("gaps", [])
    print(f"{B}{C}7. DATA GAPS{X}")
    if gaps:
        for g in gaps:
            print(f"  {Y}- {g}{X}")
    else:
        print(f"  {G}None identified{X}")
    print()

    # 8. MONITORING
    s8 = sections.get("8_monitoring", {})
    plan = s8.get("plan", {})
    print(f"{B}{C}8. MONITORING{X}")
    crp_trend = plan.get("crp_trend")
    if crp_trend:
        adm = crp_trend.get("admission_value", "?")
        cur = crp_trend.get("current_value", "?")
        pct = crp_trend.get("percent_change", "?")
        trend = crp_trend.get("trend", "?")
        sr = crp_trend.get("flag_senior_review", False)
        sr_color = R if sr else G
        print(f"  CRP: {adm} -> {cur} mg/L ({pct}% change, {trend})")
        print(f"  Senior review: {sr_color}{sr}{X}")
    tr = plan.get("treatment_response")
    if tr:
        reassess = tr.get("reassess_needed", False)
        ra_color = R if reassess else G
        print(f"  Treatment response: {ra_color}reassess_needed={reassess}{X}")
        for action in tr.get("actions", []):
            print(f"    - {action}")
    dc = plan.get("discharge_criteria_met")
    if dc is not None:
        dc_color = G if dc else R
        dc_str = "MET" if dc else "NOT MET"
        print(f"  Discharge: {dc_color}{dc_str}{X}")
    dc_detail = plan.get("discharge_criteria_details", {})
    if dc_detail:
        for check, passed_val in dc_detail.items():
            if check.startswith("_"):
                continue
            chk_color = G if passed_val else R
            chk_str = "PASS" if passed_val else "FAIL"
            print(f"    {chk_color}[{chk_str}]{X} {check}")
    print(f"  Next review: {plan.get('next_review', 'N/A')}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{B}{C}PROVENANCE{X}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{C}{'=' * 70}{X}\n")

print("render_clinical_output() defined")


In [ ]:
# Pipeline evaluation metrics
def extract_pipeline_metrics(result, scenario_name, elapsed_seconds, expected):
    """Extract metrics from a completed pipeline result.

    Args:
        result: Final pipeline state dict
        scenario_name: Human-readable scenario name
        elapsed_seconds: Wall-clock seconds for pipeline run
        expected: Dict of expected values for accuracy comparison
    """
    so = result.get("structured_output", {})
    prov = so.get("provenance", {})
    tools = prov.get("extraction_tools", {})
    curb = result.get("curb65_score", {})
    abx = result.get("antibiotic_recommendation", {})
    monitoring = result.get("monitoring_plan", {})
    contradictions = result.get("contradictions_detected", [])
    cr_ids = [c["rule_id"] for c in contradictions]

    # GPU call estimate from provenance
    gpu_ehr = 3 if tools.get("ehr") == "3-step EHR QA" else 0
    gpu_lab = 2 if tools.get("labs") == "2-step lab extraction" else 0
    gpu_cxr = 3 if tools.get("cxr") == "3-stage CXR analysis" else 0
    # Non-Strategy-E resolution count
    resolutions = result.get("resolution_results", [])
    gpu_resolution = sum(1 for r in resolutions if "(E" not in r)
    gpu_total = gpu_ehr + gpu_lab + gpu_cxr + gpu_resolution + 1  # +summary (synthesis eliminated)

    # Per-node timing from reasoning_trace
    node_times = {}
    for step in result.get("reasoning_trace", []):
        action = step.get("action", "?")
        ms = step.get("duration_ms", 0)
        node_times[action] = ms

    # Clinical accuracy checks
    accuracy = {}
    if "curb65" in expected:
        actual = curb.get("curb65")
        accuracy["curb65"] = {"expected": expected["curb65"], "actual": actual,
                              "correct": actual == expected["curb65"]}
    if "severity" in expected:
        actual = curb.get("severity_tier")
        accuracy["severity"] = {"expected": expected["severity"], "actual": actual,
                                "correct": actual == expected["severity"]}
    if "contradictions" in expected:
        for cr in expected["contradictions"]:
            accuracy[cr] = {"expected": "fired", "actual": "fired" if cr in cr_ids else "not fired",
                            "correct": cr in cr_ids}
    if "abx_contains" in expected:
        first = abx.get("first_line", "").lower()
        found = expected["abx_contains"].lower() in first
        accuracy["antibiotic"] = {"expected": expected["abx_contains"], "actual": first,
                                  "correct": found}
    if "reassess_needed" in expected:
        tr = monitoring.get("treatment_response", {})
        actual = tr.get("reassess_needed")
        accuracy["reassess_needed"] = {"expected": expected["reassess_needed"], "actual": actual,
                                       "correct": actual == expected["reassess_needed"]}
    if "crp_flagged" in expected:
        crp_t = monitoring.get("crp_trend", {})
        actual = crp_t.get("flag_senior_review")
        accuracy["crp_flagged"] = {"expected": expected["crp_flagged"], "actual": actual,
                                   "correct": actual == expected["crp_flagged"]}

    return {
        "scenario": scenario_name,
        "elapsed_s": round(elapsed_seconds, 1),
        "node_times": node_times,
        "gpu_calls": {"ehr": gpu_ehr, "lab": gpu_lab, "cxr": gpu_cxr,
                      "resolution": gpu_resolution, "total": gpu_total},
        "curb65": curb.get("curb65"),
        "severity": curb.get("severity_tier"),
        "contradictions": cr_ids,
        "first_line": abx.get("first_line", "N/A"),
        "discharge_met": monitoring.get("discharge_criteria_met"),
        "errors": len(result.get("errors", [])),
        "data_gaps": len(result.get("data_gaps", [])),
        "accuracy": accuracy,
        "ehr_pipeline": tools.get("ehr", "mock_passthrough"),
        "lab_pipeline": tools.get("labs", "mock_passthrough"),
        "cxr_pipeline": tools.get("cxr", "mock_passthrough"),
    }


def print_scenario_metrics(m):
    """Print formatted metrics for a single scenario."""
    ESC = chr(27)
    B = f"{ESC}[1m"
    C = f"{ESC}[96m"
    G = f"{ESC}[92m"
    R = f"{ESC}[91m"
    Y = f"{ESC}[93m"
    X = f"{ESC}[0m"

    print(f"\n{B}{C}PIPELINE METRICS — {m['scenario']}{X}")

    # Timing
    node_str = ", ".join(f"{k}: {v}ms" for k, v in m["node_times"].items())
    print(f"  Timing: {m['elapsed_s']}s total ({node_str})")

    # GPU calls
    gc = m["gpu_calls"]
    print(f"  GPU Calls: EHR={gc['ehr']}, Lab={gc['lab']}, CXR={gc['cxr']}, "
          f"Resolution={gc['resolution']}, Total={gc['total']}")

    # Clinical accuracy
    print(f"  {B}Clinical Accuracy:{X}")
    correct = 0
    total = 0
    for key, check in m["accuracy"].items():
        total += 1
        ok = check["correct"]
        if ok:
            correct += 1
        color = G if ok else R
        status = "CORRECT" if ok else "WRONG"
        print(f"    {color}{key}: {check['actual']} (expected: {check['expected']}) — {status}{X}")

    # Data quality
    print(f"  Data Quality: {m['data_gaps']} gaps, {m['errors']} errors")
    print(f"  Accuracy: {correct}/{total} checks correct")

print("Metrics utilities defined")


## 2. Load Model


In [ ]:
# Load MedGemma 1.5 4B (lazy singleton - only loads once)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")


## 3. Build Graph & Load Case


In [ ]:
# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")

# Load synthetic case WITH FHIR bundle for real EHR extraction
# get_synthetic_fhir_case() = flat dict + FHIR R4 Bundle
# When fhir_bundle is present, the pipeline uses real MedGemma EHR QA (3 calls)
case = get_synthetic_fhir_case()
case["cxr"]["image_path"] = CXR_CURRENT
case["cxr"]["prior_image_path"] = CXR_PRIOR
print(f"Case: {case['case_id']} - {case['demographics']['age']}yo {case['demographics']['sex']}")
print(f"Presenting: {case['presenting_complaint']}")
print(f"CXR: current={CXR_CURRENT}, prior={CXR_PRIOR}")
print(f"FHIR Bundle: {len(case['fhir_bundle']['entry'])} resources -> real EHR extraction enabled")
print(f"Lab report: {case['lab_report']['format']} from {case['lab_report']['source']} -> real lab extraction enabled")


## 4. Run Pipeline (Streaming)


In [ ]:
# Build initial state from canonical package helper
initial_state = build_initial_state(case)

print("Running CAP Agent Pipeline...")
print("=" * 60)

start_time = time.time()
result = None
prev_step = None

for chunk in graph.stream(initial_state, stream_mode="values"):
    step = chunk.get("current_step", "")
    if step and step != prev_step:
        now = time.time()
        if prev_step is not None:
            print(f"  > {prev_step} ({now - step_start:.1f}s)")
        step_start = now
        prev_step = step
    result = chunk

# Print final step timing
if prev_step is not None:
    print(f"  > {prev_step} ({time.time() - step_start:.1f}s)")

elapsed = time.time() - start_time
print("=" * 60)
print(f"Pipeline complete in {elapsed:.1f}s")

# Print progress messages
print("\n--- Pipeline Progress ---")
for msg in result.get("messages", []):
    print(f"  {msg}")


## 5. Clinician Summary


In [ ]:
print("=" * 60)
print("CLINICIAN SUMMARY")
print("=" * 60)
print(result.get("clinician_summary", "No summary generated"))


## 5.5 CXR Analysis Results


In [ ]:
from PIL import Image, ImageDraw

# Extract CXR results from pipeline output
cxr = result.get("cxr_analysis", {})
tool_results = result.get("tool_results", [])
cxr_tool = next((t for t in tool_results if t.get("tool_name") == "cxr_classification"), None)

print("=" * 60)
print("CXR ANALYSIS (3-Stage MedGemma Pipeline)")
print("=" * 60)

if cxr_tool:
    print(f"Status: {cxr_tool.get('status', 'unknown')}")
    print(f"Summary: {cxr_tool.get('summary', 'N/A')}")
else:
    print("WARNING: No CXR tool result found")

# Print classification findings
print("\n--- Classification (Stage A) ---")
for finding in ["consolidation", "pleural_effusion", "cardiomegaly", "edema", "atelectasis"]:
    f = cxr.get(finding, {})
    status = "PRESENT" if f.get("present") else "absent"
    conf = f.get("confidence", "?")
    loc = f.get("location", "")
    bbox = f.get("bounding_box")
    line = f"  {finding}: {status} ({conf} confidence)"
    if loc and loc != "not found":
        line += f" at {loc}"
    if bbox:
        line += f" [bbox: {bbox}]"
    print(line)

# Print image quality
iq = cxr.get("image_quality", {})
if iq:
    print(f"\n  Image quality: {iq.get('projection', '?')} projection, "
          f"{iq.get('rotation', '?')} rotation, {iq.get('penetration', '?')} penetration")

# Print longitudinal comparison
longit = cxr.get("longitudinal_comparison")
if longit:
    print("\n--- Longitudinal Comparison (Stage C) ---")
    for finding, change in longit.items():
        if isinstance(change, dict):
            print(f"  {finding}: {change.get('change', '?')} - {change.get('description', '')}")

# Visualize bounding boxes on CXR image
print("\n--- Localization (Stage B) ---")
try:
    draw_image = Image.open(CXR_CURRENT).copy()
    image_width, image_height = draw_image.size
    new_width = 512
    new_height = int(image_height * (new_width / image_width))
    scaled_image = draw_image.resize((new_width, new_height))
    draw = ImageDraw.Draw(scaled_image)

    bbox_count = 0
    colors = {"consolidation": "red", "pleural_effusion": "blue", "atelectasis": "green"}
    for finding_name in ["consolidation", "pleural_effusion", "atelectasis"]:
        f = cxr.get(finding_name, {})
        bbox = f.get("bounding_box")
        if bbox and len(bbox) == 4:
            y0, x0, y1, x1 = bbox
            ux0 = x0 / 1000 * new_width
            uy0 = y0 / 1000 * new_height
            ux1 = x1 / 1000 * new_width
            uy1 = y1 / 1000 * new_height
            color = colors.get(finding_name, "red")
            draw.rectangle([(ux0, uy0), (ux1, uy1)], outline=color, width=3)
            draw.text((ux0, uy0 - 15), finding_name, fill=color)
            bbox_count += 1

    if bbox_count > 0:
        print(f"  {bbox_count} finding(s) localized - see image below")
        display(scaled_image)
    else:
        print("  No localizable findings detected (no bounding boxes to draw)")
        display(scaled_image)
except Exception as e:
    print(f"  Visualization failed: {e}")


## 6. Structured Output (JSON)


In [ ]:
print("=" * 60)
print("STRUCTURED OUTPUT")
print("=" * 60)
print(json.dumps(result.get("structured_output", {}), indent=2, default=str))


In [ ]:
# Formatted clinical output (T=0)
render_clinical_output(result, "T=0 Initial Assessment")


## 7. Verification Checklist


In [ ]:
print("=" * 60)
print("VERIFICATION CHECKLIST")
print("=" * 60)

checks = []

# 1. Reasoning trace completeness
trace = result.get("reasoning_trace", [])
required_fields = ["step_number", "action", "input_summary", "output_summary", "reasoning", "timestamp", "duration_ms"]
trace_complete = all(
    all(k in step for k in required_fields)
    for step in trace
)
checks.append(("Reasoning trace: all 7 fields present in each step", trace_complete))
checks.append((f"Reasoning trace: {len(trace)} steps logged (expected >= 4)", len(trace) >= 4))

# 2. CURB65 score
curb65 = result.get("curb65_score", {})
checks.append((
    f"CURB65 score = {curb65.get('curb65')} (expected: 2)",
    curb65.get("curb65") == 2
))
checks.append((
    f"  C={curb65.get('c')} U={curb65.get('u')} R={curb65.get('r')} B={curb65.get('b')} 65={curb65.get('age_65')}",
    curb65.get("c") == 0 and curb65.get("u") == 1
    and curb65.get("r") == 0 and curb65.get("b") == 0
    and curb65.get("age_65") == 1
))

# 3. Severity tier
checks.append((
    f"Severity tier = '{curb65.get('severity_tier')}' (expected: 'moderate')",
    curb65.get("severity_tier") == "moderate"
))

# 4. CXR tool execution
cxr_tool = next((t for t in result.get("tool_results", []) if t.get("tool_name") == "cxr_classification"), None)
cxr_ran = cxr_tool is not None and cxr_tool.get("status") in ("success", "partial")
checks.append((
    f"CXR analysis: status={cxr_tool.get('status') if cxr_tool else 'missing'}",
    cxr_ran
))

# 5. CXR classification produced findings
cxr_data = result.get("cxr_analysis", {})
has_consol = "consolidation" in cxr_data
checks.append((
    f"CXR classification: consolidation key {'present' if has_consol else 'MISSING'}",
    has_consol
))

# 6. EHR extraction ran with real MedGemma
ehr_tool = next((t for t in result.get("tool_results", []) if t.get("tool_name") == "ehr_qa_extraction"), None)
ehr_ran = ehr_tool is not None and ehr_tool.get("status") in ("success", "partial")
provenance = result.get("structured_output", {}).get("provenance", {})
extraction_tools = provenance.get("extraction_tools", {})
ehr_is_real = extraction_tools.get("ehr") == "3-step EHR QA"
checks.append((
    f"EHR extraction: status={ehr_tool.get('status') if ehr_tool else 'missing'}, real={'yes' if ehr_is_real else 'mock'}",
    ehr_ran
))

# 6b. Lab extraction ran with real MedGemma
lab_tool = next((t for t in result.get("tool_results", []) if t.get("tool_name") == "lab_extraction"), None)
lab_ran = lab_tool is not None and lab_tool.get("status") in ("success", "partial")
lab_is_real = extraction_tools.get("labs") == "2-step lab extraction"
checks.append((
    f"Lab extraction: status={lab_tool.get('status') if lab_tool else 'missing'}, real={'yes' if lab_is_real else 'mock'}",
    lab_ran
))

# 7. Antibiotic recommendation
abx = result.get("antibiotic_recommendation", {})
checks.append((
    f"Antibiotic includes amoxicillin: {abx.get('first_line', 'N/A')}",
    "amoxicillin" in abx.get("first_line", "").lower()
))

# 8. All 8 sections in structured output
structured = result.get("structured_output", {})
sections = structured.get("sections", {})
checks.append((
    f"Structured output: {len(sections)}/8 sections present",
    len(sections) == 8
))

# 9. Clinician summary generated
summary = result.get("clinician_summary", "")
checks.append((
    f"Clinician summary generated ({len(summary)} chars)",
    len(summary) > 100
))

# 10. No errors
errors = result.get("errors", [])
checks.append((f"Pipeline errors = {len(errors)}", len(errors) == 0))

# 11. Pipeline completed
checks.append((
    f"Pipeline completed: current_step = '{result.get('current_step')}'",
    result.get("current_step") == "Output assembled"
))

# Print results
print()
passed = 0
failed = 0
for description, ok in checks:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}]  {description}")
    if ok:
        passed += 1
    else:
        failed += 1

# Contradictions (informational - real CXR may trigger these)
contradictions = result.get("contradictions_detected", [])
print(f"\n  [INFO] Contradictions detected: {len(contradictions)}")
for c in contradictions:
    print(f"         {c.get('rule_id', '?')}: {c.get('pattern', '')}")

print()
print(f"Results: {passed}/{passed + failed} checks passed")
if failed == 0:
    print("All verification checks passed!")
else:
    print(f"WARNING: {failed} check(s) failed")


In [ ]:
# T=0 Pipeline Metrics
metrics_t0 = extract_pipeline_metrics(result, "T=0 Initial Assessment", elapsed, {
    "curb65": 2, "severity": "moderate", "abx_contains": "amoxicillin"
})
print_scenario_metrics(metrics_t0)


## 8. 48 Hours Later — Antibiotic Stewardship Review

**Clinical Progress:** Blood cultures returned **Streptococcus pneumoniae** (penicillin-sensitive). The patient is clinically improving — temperature normalised, eating independently, AMT 10/10. Currently on Co-amoxiclav 1.2g TDS IV + Clarithromycin 500mg BD IV (52 hours of IV therapy).

**What we're testing:** The pipeline re-runs on **updated FHIR data** (T=48h vitals, labs, microbiology) to detect antibiotic stewardship opportunities:
- **CR-8:** Macrolide prescribed but no atypical pathogen on microbiology → de-escalation recommended
- **CR-9:** IV route >48h but patient clinically stable + tolerating oral → IV-to-oral switch recommended
- **CR-7:** Should NOT fire — S. pneumoniae is susceptible to current antibiotics

> This demonstrates **temporal clinical reasoning** — the same pipeline, different timepoint, new stewardship insights.


In [ ]:
from cap_agent.data.synthetic import get_synthetic_48h_case

case_48h = get_synthetic_48h_case()
# No CXR image assignment — clinically, CXR is not repeated at 48h
# The pipeline will use mock CXR passthrough (0 GPU calls for CXR)
initial_state_48h = build_initial_state(case_48h)

print("Running pipeline at T=48h (stewardship review)...")
print(f"  Micro results: {len(case_48h.get('micro_results', []))} culture(s)")
print(f"  Treatment: {case_48h['treatment_status']['current_route']} for {case_48h['treatment_status']['hours_on_iv']}h")
print(f"  FHIR bundle: {len(case_48h['fhir_bundle']['entry'])} resources (updated for T=48h)")
print()

start_48h = time.time()
result_48h = None
prev_step = None

for chunk in graph.stream(initial_state_48h, stream_mode="values"):
    step = chunk.get("current_step", "")
    if step and step != prev_step:
        now = time.time()
        if prev_step is not None:
            print(f"  > {prev_step} ({now - step_start:.1f}s)")
        step_start = now
        prev_step = step
    result_48h = chunk

if prev_step is not None:
    print(f"  > {prev_step} ({time.time() - step_start:.1f}s)")

elapsed_48h = time.time() - start_48h
print(f"\nT=48h pipeline complete in {elapsed_48h:.1f}s")


In [ ]:
print("=" * 60)
print("T=48h CONTRADICTIONS DETECTED")
print("=" * 60)
for c in result_48h.get("contradictions_detected", []):
    print(f"  [{c['rule_id']}] {c['pattern']}")
    if c.get("recommendation"):
        print(f"    >>> {c['recommendation']}")
print()

print("=" * 60)
print("T=48h RESOLUTIONS")
print("=" * 60)
for r in result_48h.get("resolution_results", []):
    print(f"  {r}")
print()

score = result_48h.get("curb65_score", {})
print(f"CURB65 at T=48h: {score.get('curb65')} ({score.get('severity_tier')} severity)")
print(f"  (T=0 was CURB65=2, moderate -> now {score.get('curb65')}, {score.get('severity_tier')})")


In [ ]:
print("=" * 60)
print("T=48h CLINICIAN SUMMARY")
print("=" * 60)
print(result_48h.get("clinician_summary", "No summary generated"))


In [ ]:
# Formatted clinical output (T=48h)
render_clinical_output(result_48h, "T=48h Stewardship Review")


In [ ]:
print("=" * 60)
print("T=48h VERIFICATION CHECKLIST")
print("=" * 60)

checks_48h = []

# 1. CR-8 fired (macrolide de-escalation)
contradictions_48h = result_48h.get("contradictions_detected", [])
cr_ids = [c["rule_id"] for c in contradictions_48h]
cr8_fired = "CR-8" in cr_ids
checks_48h.append(("CR-8 fired (macrolide de-escalation)", cr8_fired))

# 2. CR-9 fired (IV-to-oral switch)
cr9_fired = "CR-9" in cr_ids
checks_48h.append(("CR-9 fired (IV-to-oral switch)", cr9_fired))

# 3. CR-9 has recommendation
cr9_alerts = [c for c in contradictions_48h if c["rule_id"] == "CR-9"]
cr9_has_rec = bool(cr9_alerts and cr9_alerts[0].get("recommendation"))
checks_48h.append(("CR-9 includes IV-to-oral recommendation", cr9_has_rec))

# 4. CR-7 did NOT fire (organism susceptible)
cr7_absent = "CR-7" not in cr_ids
checks_48h.append(("CR-7 did not fire (organism susceptible)", cr7_absent))

# 5. Strategy E resolutions present
resolutions_48h = result_48h.get("resolution_results", [])
has_e_resolutions = any("(E " in r for r in resolutions_48h)
checks_48h.append(("Strategy E deterministic resolutions present", has_e_resolutions))

# 6. CURB65 improved
score_48h = result_48h.get("curb65_score", {})
curb_improved = (score_48h.get("curb65") is not None and score_48h.get("curb65") <= 1)
checks_48h.append(("CURB65 improved to low severity", curb_improved))

# 7. No errors
no_errors = len(result_48h.get("errors", [])) == 0
checks_48h.append(("No pipeline errors", no_errors))

# 8. Pipeline completed
completed = result_48h.get("current_step") == "Output assembled"
checks_48h.append(("Pipeline completed", completed))

print()
passed = sum(1 for _, ok in checks_48h if ok)
for label, ok in checks_48h:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {label}")
print(f"\nT=48h: {passed}/{len(checks_48h)} checks passed")
if passed == len(checks_48h):
    print("All T=48h stewardship checks passed!")


In [ ]:
# T=48h Pipeline Metrics
metrics_48h = extract_pipeline_metrics(result_48h, "T=48h Stewardship Review", elapsed_48h, {
    "severity": "low", "contradictions": ["CR-8", "CR-9"], "abx_contains": "amoxicillin"
})
print_scenario_metrics(metrics_48h)


## 9. Safety Case — Fluoroquinolone Prescribing Review (CR-10)

**Different patient scenario:** A high-severity CAP patient (CURB65=3) with a documented **penicillin allergy** — but the recorded reaction is **GI upset** (an intolerance, not a true allergy).

**Expected behavior:**
1. High severity + penicillin allergy → system prescribes **levofloxacin** (evidence-based selection)
2. **CR-10 fires:** Detects that the "allergy" is actually an intolerance (GI upset)
3. Recommends reassessing whether a **beta-lactam** could still be used safely
4. Includes **MHRA fluoroquinolone safety warning** (tendon/aortic risk)

> This demonstrates the system's ability to catch unnecessary fluoroquinolone exposure — a key antimicrobial stewardship concern and patient safety issue.


In [ ]:
from cap_agent.data.synthetic import get_synthetic_cr10_case

case_cr10 = get_synthetic_cr10_case()
# No CXR images, no FHIR — all mock extraction (fast demo)
initial_state_cr10 = build_initial_state(case_cr10)

print("Running pipeline — CR-10 safety demo (high severity + penicillin intolerance)...")
print(f"  Allergies: {case_cr10['demographics'].get('allergies', case_cr10['past_medical_history']['allergies'])}")
print()

start_cr10 = time.time()
result_cr10 = None
prev_step = None

for chunk in graph.stream(initial_state_cr10, stream_mode="values"):
    step = chunk.get("current_step", "")
    if step and step != prev_step:
        now = time.time()
        if prev_step is not None:
            print(f"  > {prev_step} ({now - step_start:.1f}s)")
        step_start = now
        prev_step = step
    result_cr10 = chunk

if prev_step is not None:
    print(f"  > {prev_step} ({time.time() - step_start:.1f}s)")

elapsed_cr10 = time.time() - start_cr10
print(f"\nCR-10 pipeline complete in {elapsed_cr10:.1f}s")
print()

print("=" * 60)
print("CR-10 CONTRADICTIONS")
print("=" * 60)
for c in result_cr10.get("contradictions_detected", []):
    print(f"  [{c['rule_id']}] {c['pattern']}")
    if c.get("recommendation"):
        print(f"    >>> {c['recommendation']}")


In [ ]:
# Formatted clinical output (CR-10)
render_clinical_output(result_cr10, "CR-10 Safety Demo")


In [ ]:
print("=" * 60)
print("CR-10 VERIFICATION CHECKLIST")
print("=" * 60)

checks_cr10 = []

# 1. CR-10 fired
contradictions_cr10 = result_cr10.get("contradictions_detected", [])
cr10_ids = [c["rule_id"] for c in contradictions_cr10]
cr10_fired = "CR-10" in cr10_ids
checks_cr10.append(("CR-10 fired (fluoroquinolone + penicillin intolerance)", cr10_fired))

# 2. CURB65 is high severity
score_cr10 = result_cr10.get("curb65_score", {})
high_severity = score_cr10.get("severity_tier") == "high"
checks_cr10.append(("CURB65 severity is high", high_severity))

# 3. Levofloxacin in treatment
abx_cr10 = result_cr10.get("antibiotic_recommendation", {})
has_levo = "levofloxacin" in str(abx_cr10).lower()
checks_cr10.append(("Levofloxacin prescribed (triggers CR-10)", has_levo))

# 4. No pipeline errors
no_errors = len(result_cr10.get("errors", [])) == 0
checks_cr10.append(("No pipeline errors", no_errors))

# 5. Pipeline completed
completed = result_cr10.get("current_step") == "Output assembled"
checks_cr10.append(("Pipeline completed", completed))

print()
passed = sum(1 for _, ok in checks_cr10 if ok)
for label, ok in checks_cr10:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {label}")
print(f"\nCR-10: {passed}/{len(checks_cr10)} checks passed")
if passed == len(checks_cr10):
    print("All CR-10 safety checks passed!")


In [ ]:
# CR-10 Pipeline Metrics
metrics_cr10 = extract_pipeline_metrics(result_cr10, "CR-10 Safety Demo", elapsed_cr10, {
    "severity": "high", "contradictions": ["CR-10"], "abx_contains": "levofloxacin"
})
print_scenario_metrics(metrics_cr10)


## 10. Treatment Monitoring — Day 3-4 Reassessment

**Same patient** (Robert James, 72yo), now **Day 3 of treatment** (13/02/2026).

**Clinical scenario — NOT improving:**
- Switched to oral amoxicillin at 48h (per CR-8/CR-9 stewardship)
- Low-grade fever persisting (37.8C), productive cough continues
- CRP only decreased 41% from admission (186 → 110 mg/L) — **under the 50% threshold**
- S. pneumoniae confirmed on both blood and sputum cultures

**What this demo tests:**
1. **CRP trend analysis** — `compute_crp_trend(186, 110)` flags <50% decrease for senior review
2. **Treatment response assessment** — Day 3 without improvement triggers reassessment
3. Both results surface in the monitoring plan output

**Expected pipeline behavior:**
- CURB65 = 1 (low): urea 7.0 (U=0, strictly >7), age 72 (65=1)
- CRP trend: slow_response, flag_senior_review=True
- Treatment response: reassess_needed=True, 5 recommended actions
- EHR + Lab extraction run real MedGemma pipelines; CXR uses mock (not repeated)

> This demonstrates temporal reasoning: the system doesn't just assess the current state, it compares against the admission baseline to detect inadequate treatment response.


In [ ]:
from cap_agent.data.synthetic import get_synthetic_day34_case

case_day34 = get_synthetic_day34_case()
initial_state_day34 = build_initial_state(case_day34)

print("Running pipeline — Day 3-4 treatment monitoring (NOT improving)...")
print(f"  Treatment: {case_day34['treatment_status']}")
print(f"  Admission CRP: {case_day34['admission_labs']['crp']} mg/L")
print(f"  Current CRP: {case_day34['lab_results']['crp']['value']} mg/L")
print()

start_day34 = time.time()
result_day34 = None
prev_step = None

for chunk in graph.stream(initial_state_day34, stream_mode="values"):
    step = chunk.get("current_step", "")
    if step and step != prev_step:
        now = time.time()
        if prev_step is not None:
            print(f"  > {prev_step} ({now - step_start:.1f}s)")
        step_start = now
        prev_step = step
    result_day34 = chunk

if prev_step is not None:
    print(f"  > {prev_step} ({time.time() - step_start:.1f}s)")

elapsed_day34 = time.time() - start_day34
print(f"\nDay 3-4 pipeline complete in {elapsed_day34:.1f}s")


In [ ]:
print("=" * 60)
print("DAY 3-4 MONITORING RESULTS")
print("=" * 60)

monitoring_day34 = result_day34.get("monitoring_plan", {})

# CRP Trend
crp_trend = monitoring_day34.get("crp_trend")
if crp_trend:
    print("\n--- CRP Trend Analysis ---")
    print(f"  Admission CRP: {crp_trend['admission_value']} mg/L")
    print(f"  Current CRP:   {crp_trend['current_value']} mg/L")
    print(f"  Change:         {crp_trend['percent_change']}% decrease")
    print(f"  Trend:          {crp_trend['trend']}")
    print(f"  Senior review:  {crp_trend['flag_senior_review']}")
    print(f"  Reasoning:      {crp_trend['reasoning']}")
else:
    print("\n  No CRP trend data available")

# Treatment Response
treatment_response = monitoring_day34.get("treatment_response")
if treatment_response:
    print("\n--- Treatment Response Assessment ---")
    print(f"  Reassess needed: {treatment_response['reassess_needed']}")
    print(f"  Reasoning:       {treatment_response['reasoning']}")
    if treatment_response.get("actions"):
        print("  Actions:")
        for action in treatment_response["actions"]:
            print(f"    - {action}")
else:
    print("\n  No treatment response data available")

# Monitoring plan details
print("\n--- Monitoring Plan ---")
print(f"  CRP timing: {monitoring_day34.get('crp_repeat_timing', 'N/A')[:120]}...")
print(f"  Next review: {monitoring_day34.get('next_review', 'N/A')}")
print(f"  Discharge criteria met: {monitoring_day34.get('discharge_criteria_met')}")


In [ ]:
print("=" * 60)
print("DAY 3-4 CLINICIAN SUMMARY")
print("=" * 60)
print()
print(result_day34.get("clinician_summary", "No summary generated"))


In [ ]:
# Formatted clinical output (Day 3-4)
render_clinical_output(result_day34, "Day 3-4 Treatment Monitoring")


In [ ]:
print("=" * 60)
print("DAY 3-4 VERIFICATION CHECKLIST")
print("=" * 60)

checks_day34 = []

# 1. Treatment reassessment triggered
monitoring_d34 = result_day34.get("monitoring_plan", {})
treatment_resp = monitoring_d34.get("treatment_response")
reassess = treatment_resp is not None and treatment_resp.get("reassess_needed") is True
checks_day34.append(("Treatment reassessment triggered (Day 3, not improving)", reassess))

# 2. CRP trend flagged for senior review
crp_t = monitoring_d34.get("crp_trend")
crp_flagged = crp_t is not None and crp_t.get("flag_senior_review") is True
checks_day34.append(("CRP trend flagged for senior review (<50% decrease)", crp_flagged))

# 3. CRP percent_change in expected range (~41%)
pct = crp_t.get("percent_change", 0) if crp_t else 0
pct_ok = 35 <= pct <= 45
checks_day34.append((f"CRP percent_change ~41% (got {pct}%)", pct_ok))

# 4. EHR extraction ran (real, via FHIR)
tool_results = result_day34.get("tool_results", [])
ehr_ran = any(t.get("tool_name") == "ehr_qa_extraction" for t in tool_results)
checks_day34.append(("EHR extraction ran", ehr_ran))

# 5. Lab extraction ran (real, via lab_report)
lab_ran = any(t.get("tool_name") == "lab_extraction" for t in tool_results)
checks_day34.append(("Lab extraction ran", lab_ran))

# 6. No pipeline errors
no_errors = len(result_day34.get("errors", [])) == 0
checks_day34.append(("No pipeline errors", no_errors))

# 7. Pipeline completed
completed = result_day34.get("current_step") == "Output assembled"
checks_day34.append(("Pipeline completed", completed))

print()
passed = sum(1 for _, ok in checks_day34 if ok)
for label, ok in checks_day34:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {label}")
print(f"\nDay 3-4: {passed}/{len(checks_day34)} checks passed")
if passed == len(checks_day34):
    print("All Day 3-4 treatment monitoring checks passed!")


In [ ]:
# Day 3-4 Pipeline Metrics
metrics_day34 = extract_pipeline_metrics(result_day34, "Day 3-4 Treatment Monitoring", elapsed_day34, {
    "severity": "low", "abx_contains": "amoxicillin", "reassess_needed": True, "crp_flagged": True
})
print_scenario_metrics(metrics_day34)


In [ ]:
# Cross-scenario comparison table
ESC = chr(27)
B = f"{ESC}[1m"; C = f"{ESC}[96m"; G = f"{ESC}[92m"; R = f"{ESC}[91m"; X = f"{ESC}[0m"

all_metrics = [metrics_t0, metrics_48h, metrics_cr10, metrics_day34]

print(f"\n{B}{C}{'=' * 80}")
print(f"  CROSS-SCENARIO COMPARISON")
print(f"{'=' * 80}{X}\n")

hdr = f"  {'Metric':<22} {'T=0 Initial':>13} {'T=48h Review':>13} {'CR-10 Safety':>13} {'Day 3-4':>13}"
print(hdr)
print(f"  {'─' * 74}")

def _v(m, k):
    v = m.get(k)
    return str(v)[:13] if v is not None else "N/A"

def _cr(m):
    crs = m.get("contradictions", [])
    return ",".join(crs) if crs else "None"

def _gpu(m):
    return str(m.get("gpu_calls", {}).get("total", "?"))

rows = [
    ("CURB65",         [_v(m, "curb65") for m in all_metrics]),
    ("Severity",       [_v(m, "severity") for m in all_metrics]),
    ("Contradictions", [_cr(m) for m in all_metrics]),
    ("First-line Abx", [_v(m, "first_line") for m in all_metrics]),
    ("Discharge met",  [_v(m, "discharge_met") for m in all_metrics]),
    ("Pipeline (s)",   [_v(m, "elapsed_s") for m in all_metrics]),
    ("GPU calls",      [_gpu(m) for m in all_metrics]),
    ("Errors",         [_v(m, "errors") for m in all_metrics]),
    ("EHR pipeline",   [_v(m, "ehr_pipeline") for m in all_metrics]),
    ("Lab pipeline",   [_v(m, "lab_pipeline") for m in all_metrics]),
    ("CXR pipeline",   [_v(m, "cxr_pipeline") for m in all_metrics]),
]
for label, vals in rows:
    print(f"  {label:<22} {vals[0]:>13} {vals[1]:>13} {vals[2]:>13} {vals[3]:>13}")

total_correct = sum(sum(1 for c in m["accuracy"].values() if c["correct"]) for m in all_metrics)
total_checks = sum(len(m["accuracy"]) for m in all_metrics)
total_gpu = sum(m["gpu_calls"]["total"] for m in all_metrics)
total_time = sum(m["elapsed_s"] for m in all_metrics)

print(f"\n  {B}Clinical accuracy: {total_correct}/{total_checks} checks correct{X}")
print(f"  {B}Total GPU calls: {total_gpu}{X}")
print(f"  {B}Total pipeline time: {total_time:.1f}s{X}")
print(f"\n{C}{'=' * 80}{X}")
